LLMs API Call

In [13]:
import os
from google.colab import userdata
PROVIDER = 'gemini'

prompt = "Explain what an LLM is in 2 sentences, as if to a 10-year-old."

if PROVIDER == 'gemini':
  import google.generativeai as genai
  google_api_key = userdata.get('GEMINI_API_KEY')
  os.environ['GOOGLE_API_KEY'] = google_api_key
  genai.configure(api_key=google_api_key)
  model = genai.GenerativeModel('gemini-3.6-flash')
  response = model.generate_content(prompt)
  print(f"Gemini 3 Flash: {response.text}")

elif PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY)
    resp = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    print("GPT-5 nano:", resp.choices[0].message.content)

elif PROVIDER == "anthropic":
    from anthropic import Anthropic
    client = Anthropic(api_key=API_KEY)
    resp = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    print("Claude Sonnet 5:", resp.content[0].text)

Gemini 3 Flash: An LLM is like a super-smart computer brain that learned how to talk by reading millions of books and websites. Because it knows so many words, it can answer your questions, help with homework, and write fun stories with you!


Token counting and cost estimation

In [16]:
import tiktoken

enc = tiktoken.get_encoding('o200k_base')
text = """Large Language Models predict the next token.
          Train this on internet-scale data, and systems emerge
          that write code, pass exams, and explain science
        """
tokens = enc.encode(text)
print(f'Text: {len(text)} characters')
print(f'Tokens: {len(tokens)} tokens')
print(f'Ratio: {len(text)/len(tokens)} per token')


for i, tok in enumerate(tokens[:10]):
    print(f"  Token {i}: {tok} -> '{enc.decode([tok])}'")
if len(tokens) > 10:
    print(f"  ... ({len(tokens) - 10} more tokens)")


# Cost Estimation

# Approximate prices per 1M tokens (2026)
PRICES = {
    "GPT-5.6 Sol (frontier)": {"input": 5.00,  "output": 30.00},
    "GPT-5.6 Luna (speed)":   {"input": 1.00,  "output":  6.00},
    "Claude Opus 5":        {"input": 5.00,  "output": 25.00},
    "Claude Sonnet 5":        {"input": 3.00,  "output": 15.00},
    "Claude Haiku 4.5":       {"input": 1.00,  "output":  5.00},
    "Gemini 3.6 Flash":       {"input": 1.50,  "output":  7.50},
    "Gemini 3.5 Flash-Lite":  {"input": 0.30,  "output":  2.50},
    "Llama 4 (self-host)":    {"input": 0.00,  "output":  0.00},
}

# Simulate: 500 token prompt, 200 token response, 10K queries/day
prompt_tokens = 500
response_tokens = 200
queries_per_day = 10_000

print("\n" + "=" * 60)
print("DAILY COST: 10K queries (500 in + 200 out tokens each)")
print("=" * 60)
for model, prices in PRICES.items():
    daily_in = (prompt_tokens * queries_per_day / 1_000_000) * prices["input"]
    daily_out = (response_tokens * queries_per_day / 1_000_000) * prices["output"]
    daily = daily_in + daily_out
    monthly = daily * 30
    print(f"{model:25s}: ${daily:8.2f}/day  ${monthly:10.2f}/month")

Text: 177 characters
Tokens: 33 tokens
Ratio: 5.363636363636363 per token
  Token 0: 42565 -> 'Large'
  Token 1: 20333 -> ' Language'
  Token 2: 50258 -> ' Models'
  Token 3: 17946 -> ' predict'
  Token 4: 290 -> ' the'
  Token 5: 2613 -> ' next'
  Token 6: 6602 -> ' token'
  Token 7: 558 -> '.
'
  Token 8: 983 -> '         '
  Token 9: 32131 -> ' Train'
  ... (23 more tokens)

DAILY COST: 10K queries (500 in + 200 out tokens each)
GPT-5.6 Sol (frontier)   : $   85.00/day  $   2550.00/month
GPT-5.6 Luna (speed)     : $   17.00/day  $    510.00/month
Claude Opus 5            : $   75.00/day  $   2250.00/month
Claude Sonnet 5          : $   45.00/day  $   1350.00/month
Claude Haiku 4.5         : $   15.00/day  $    450.00/month
Gemini 3.6 Flash         : $   22.50/day  $    675.00/month
Gemini 3.5 Flash-Lite    : $    6.50/day  $    195.00/month
Llama 4 (self-host)      : $    0.00/day  $      0.00/month
